<a href="https://colab.research.google.com/github/DangHuuLong/Ai-Recruiter-Mini-Ai-Service/blob/experiment%2Fcross-encoder-v0.1/notebooks/fine_tune_cross_encoder_v0.1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Cross-Encoder CV-JD v0.1

Fine-tune `cross-encoder/ms-marco-MiniLM-L-6-v2` trên 4900 CV-JD pairs sử dụng MSE regression.
Sau khi train: `model.predict(pairs) * 100` → calibrated score 0–100.

| | |
|---|---|
| **Dataset** | v0.3 — 4900 train / 1050 validation / 1050 test |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| **Loss** | MSE regression, label = score / 100, sigmoid activation |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.1` |


In [4]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.1
!git pull origin experiment/cross-encoder-v0.1


/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/cross-encoder-v0.1'
Your branch is up to date with 'origin/experiment/cross-encoder-v0.1'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 466 bytes | 466.00 KiB/s, done.
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.1 -> FETCH_HEAD
   4562186..c3ac72a  experiment/cross-encoder-v0.1 -> origin/experiment/cross-encoder-v0.1
Updating 4562186..c3ac72a
Fast-forward
 training/fine_tune_cross_encoder.py | 5 ++++-
 1 file changed, 4 insertions(+), 1 deletion(-)


## 1. Install dependencies


In [4]:
!pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2

## 2. Validate data

Kiểm tra JSONL files cross-encoder đã đủ 3 splits chưa.


In [5]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.3/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")


train       :  4900 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


## 3. Debug run

Chạy 1 epoch với 40 train / 20 val để xác nhận pipeline không lỗi trước khi train full.


In [6]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20


2026-06-17 15:33:14.608528: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-6-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : artifacts/models/cross-encoder-cv-jd-v0.1
Epochs     : 1  |  Batch size: 4  |  Max length: 512

Train: 40 pairs  |  Val: 20 pairs

Steps/epoch: 10  |  Warmup steps: 1

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  epoch  1/1  step    10  val_spearman=-0.3506
  [done]  val_spearman=-0.3506

Fine-tuning complete.
Model written to: artifacts/models/

## 4. Full training

Model lưu vào `ai-recruiter/models/` trên Drive, report lưu local rồi copy sang `ai-recruiter/reports/`.


In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Device: Tesla T4


In [8]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.1 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.1_report.json \
    --epochs 10 \
    --batch-size 16


Mounted at /content/drive
2026-06-17 15:34:15.638488: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-6-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : /content/drive/MyDrive/ai-recruiter/models/cross-encoder-cv-jd-v0.1
Epochs     : 10  |  Batch size: 16  |  Max length: 512

Train: 4900 pairs  |  Val: 1050 pairs

Steps/epoch: 307  |  Warmup steps: 30

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  epoch  1/10  step   307  val_spearman=0.8322
  [done]  val_spearman=0.8322

## 5. Results


In [9]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.1_report.json").read_text(encoding="utf-8")
)
print(json.dumps(report["metrics"], indent=2))


{
  "validation": {
    "mae": 10.2374,
    "rmse": 13.9753,
    "label_accuracy": 0.5257,
    "pair_count": 1050,
    "mean_predicted_score": 61.2753,
    "mean_target_score": 60.6457
  },
  "test": {
    "mae": 10.0622,
    "rmse": 14.7028,
    "label_accuracy": 0.5924,
    "pair_count": 1050,
    "mean_predicted_score": 61.7817,
    "mean_target_score": 62.8514
  }
}


In [10]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.1_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.1_report.json",
)
print(f"Saved to {reports_dir}")


Saved to /content/drive/MyDrive/ai-recruiter/reports
